1. Post-processing — Car-only emissions for peri-urban residents 

In [1]:
from __future__ import annotations

from pathlib import Path
import csv
import re
import xml.etree.ElementTree as ET
import logging

import numpy as np
import pandas as pd

# =============================================================================
# Post-processing — Car-only emissions for peri-urban residents (CSV outputs)
# =============================================================================


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
PERSONS_CSV = Path("../Eqasim_output/17persons_filtered.csv")          # must contain: person_id, household_id
HOUSEHOLDS_CSV = Path("../Eqasim_output/17households_filtered.csv")    # must contain: household_id, commune_id

ROU_XML = Path("../5-vehicules/population_all_with_vtypes.rou.xml")
TRIPINFO_XML = Path("../6-simulation/tripinfo.xml")  

OUT_DIR = Path("outputs_emissions_car_only")

PERIURBAN_COMMUNES_8 = {17059, 17245, 17109, 17194, 17373, 17315, 17136, 17420}

DEPART_TOL_SEC = 120
ROLE_FACTOR = {"car": 1.0, "car_passenger": 0.5}
TOP_N = 50

# Logging (repository-friendly)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("emissions_postproc")


# -----------------------------------------------------------------------------
# I/O utilities
# -----------------------------------------------------------------------------
def ensure_dir(path: Path) -> None:
    """Create output directory if needed."""
    path.mkdir(parents=True, exist_ok=True)

def read_csv_semicolon(path: Path) -> pd.DataFrame:
    """Read pipeline CSV files (semicolon-separated)."""
    return pd.read_csv(path, sep=";", low_memory=False)

def repair_tripinfo_if_needed(path: Path) -> Path:
    """
    Repair tripinfo.xml if the closing </tripinfos> tag is missing.
    ElementTree may fail on truncated XML.
    """
    if not path.exists():
        raise FileNotFoundError(f"tripinfo file not found: {path.resolve()}")

    data = path.read_bytes()
    if b"</tripinfos>" in data[-50000:]:
        return path

    if b"<tripinfos" not in data[:200000]:
        raise RuntimeError("Input does not look like a SUMO tripinfo output (<tripinfos ...> not found).")

    repaired = path.with_suffix(".repaired.xml")
    repaired.write_bytes(data + b"\n</tripinfos>\n")
    logger.warning("tripinfo appears truncated -> repaired file written: %s", repaired)
    return repaired


# -----------------------------------------------------------------------------
# Step 1 — Identify residents (8 communes) using households commune_id
# -----------------------------------------------------------------------------
def load_resident_person_ids(persons_csv: Path, households_csv: Path, communes_8: set[int]) -> set[str]:
    """
    Return the set of person_id that belong to households located in the target communes.
    """
    if not persons_csv.exists():
        raise FileNotFoundError(f"persons CSV not found: {persons_csv.resolve()}")
    if not households_csv.exists():
        raise FileNotFoundError(f"households CSV not found: {households_csv.resolve()}")

    persons = read_csv_semicolon(persons_csv)
    households = read_csv_semicolon(households_csv)

    persons["person_id"] = persons["person_id"].astype(str)
    persons["household_id"] = persons["household_id"].astype(str)

    households["household_id"] = households["household_id"].astype(str)
    if "commune_id" not in households.columns:
        raise KeyError(f"{households_csv} is missing required column: 'commune_id'")

    households["commune_id"] = pd.to_numeric(households["commune_id"], errors="coerce").astype("Int64")
    hh_ids = set(households.loc[households["commune_id"].isin(communes_8), "household_id"].astype(str))

    resident_ids = set(persons.loc[persons["household_id"].isin(hh_ids), "person_id"].astype(str))
    logger.info("Residents (8 communes): %d persons", len(resident_ids))
    return resident_ids


# -----------------------------------------------------------------------------
# Step 2 — Extract car personTrips (+ role) from routes file
# -----------------------------------------------------------------------------
def is_car_mode(modes_value: str) -> bool:
    """
    personTrip@modes can be 'car' or 'car taxi' etc.
    We treat it as car if 'car' is present as a token.
    """
    if not modes_value:
        return False
    tokens = modes_value.replace(",", " ").split()
    return "car" in tokens

def extract_car_trips_from_rou(rou_path: Path, keep_person_ids: set[str]) -> pd.DataFrame:
    """
    Return a DataFrame with one row per car personTrip:
      - person_id
      - trip_index (car trip counter per person)
      - role (car or car_passenger)
      - factor (ROLE_FACTOR applied)
      - expected_depart (estimated from stop until times)
    """
    if not rou_path.exists():
        raise FileNotFoundError(f"ROU file not found: {rou_path.resolve()}")

    tree = ET.parse(rou_path)
    root = tree.getroot()

    rows = []
    for person in root.findall("person"):
        pid = (person.get("id") or "").strip()
        if not pid or pid not in keep_person_ids:
            continue

        t = float(person.get("depart", "0") or 0.0)
        car_trip_idx = 0

        for elem in list(person):
            if elem.tag == "stop":
                until = elem.get("until")
                if until is not None:
                    t = float(until)

            elif elem.tag == "personTrip":
                if is_car_mode(elem.get("modes", "")):
                    role = elem.get("role", "car")
                    factor = float(ROLE_FACTOR.get(role, 1.0))
                    rows.append(
                        {
                            "person_id": pid,
                            "trip_index": car_trip_idx,
                            "role": role,
                            "factor": factor,
                            "expected_depart": t,
                        }
                    )
                    car_trip_idx += 1

    df = pd.DataFrame(rows)
    logger.info("Car personTrips extracted from ROU: %d", len(df))
    return df


# -----------------------------------------------------------------------------
# Step 3 — Stream-read tripinfo and keep private vehicles only (exclude vaporized)
# -----------------------------------------------------------------------------
PRIVATE_VEH_RE = re.compile(r"^(\d+)_(\d+)$")  # e.g. "12345_0" -> person_id "12345"

def vehicle_to_person_from_private_id(veh_id: str) -> str:
    m = PRIVATE_VEH_RE.match(str(veh_id))
    return m.group(1) if m else ""

def iter_tripinfo_rows(tripinfo_path: Path):
    """
    Streaming parser (low RAM): yields one dict per <tripinfo> element.
    Emissions are read from child node <emissions .../> if present.
    """
    context = ET.iterparse(tripinfo_path, events=("end",))
    for _event, elem in context:
        if elem.tag != "tripinfo":
            continue

        row = dict(elem.attrib)
        em = elem.find("emissions")
        if em is not None:
            for k, v in em.attrib.items():
                row.setdefault(k, v)

        yield row
        elem.clear()

def parse_tripinfo_filtered(tripinfo_path: Path, keep_person_ids: set[str]) -> pd.DataFrame:
    """
    Build a DataFrame of tripinfo rows restricted to:
      - private vehicles with IDs matching "<person_id>_<idx>"
      - person_id in keep_person_ids
      - not vaporized
    """
    rows = []
    for row in iter_tripinfo_rows(tripinfo_path):
        veh_id = row.get("id", "")
        pid = vehicle_to_person_from_private_id(veh_id)

        if not pid:
            continue
        if pid not in keep_person_ids:
            continue

        vap = str(row.get("vaporized", "")).strip().lower()
        if vap in {"true", "1", "yes"}:
            continue

        row["vehicle_id"] = str(veh_id)
        row["person_id"] = str(pid)
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Convert numeric columns (keep a few identifiers as strings)
    skip = {"id", "vehicle_id", "person_id", "vType", "type", "departLane", "arrivalLane", "devices", "vaporized"}
    for c in df.columns:
        if c in skip:
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")

    logger.info("tripinfo rows kept (private + residents): %d", len(df))
    return df


# -----------------------------------------------------------------------------
# Step 4 — Matching + allocation + aggregation + CSV outputs
# -----------------------------------------------------------------------------
def summary_stats(series: pd.Series) -> dict:
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return dict(mean=np.nan, median=np.nan, std=np.nan, p10=np.nan, p90=np.nan, min=np.nan, max=np.nan)
    return dict(
        mean=float(s.mean()),
        median=float(s.median()),
        std=float(s.std(ddof=1)) if len(s) > 1 else 0.0,
        p10=float(np.percentile(s, 10)),
        p90=float(np.percentile(s, 90)),
        min=float(s.min()),
        max=float(s.max()),
    )

def gini(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    if np.min(x) < 0:
        x = x - np.min(x)
    total = x.sum()
    if np.isclose(total, 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n)

def run() -> None:
    ensure_dir(OUT_DIR)

    # A) Residents
    resident_ids = load_resident_person_ids(PERSONS_CSV, HOUSEHOLDS_CSV, PERIURBAN_COMMUNES_8)

    # B) Planned car trips + roles from ROU
    df_role = extract_car_trips_from_rou(ROU_XML, resident_ids)
    if df_role.empty:
        raise RuntimeError("No car personTrips found in ROU for the selected residents.")

    # C) Observed tripinfo rows
    fixed_tripinfo = repair_tripinfo_if_needed(TRIPINFO_XML)
    df_trip = parse_tripinfo_filtered(fixed_tripinfo, resident_ids)
    if df_trip.empty:
        raise RuntimeError("No private vehicle <tripinfo> rows found after filtering.")

    # Emissions columns depend on SUMO config/version
    possible_em_cols = [
        "CO2_abs", "CO_abs", "HC_abs", "NOx_abs", "PMx_abs", "fuel_abs", "electricity_abs",
        "CO2", "CO", "HC", "NOx", "PMx", "fuel", "electricity", "FC_MJ"
    ]
    em_cols = [c for c in possible_em_cols if c in df_trip.columns]
    if not em_cols:
        raise RuntimeError("No emissions columns found in tripinfo.xml (device.emissions enabled?).")

    dist_col = "routeLength" if "routeLength" in df_trip.columns else None
    dur_col = "duration" if "duration" in df_trip.columns else None

    # D) Matching tripinfo <-> personTrips per person using closest depart time
    matched_rows = []
    mismatch = []

    for pid, g_role in df_role.groupby("person_id"):
        g_role = g_role.sort_values("expected_depart").reset_index(drop=True)

        g_trip = df_trip[df_trip["person_id"] == pid].copy()
        if g_trip.empty:
            mismatch.append((pid, len(g_role), 0, "no_tripinfo_for_person"))
            continue

        has_depart = ("depart" in g_trip.columns) and g_trip["depart"].notna().any()
        if has_depart:
            g_trip = g_trip.sort_values("depart").reset_index(drop=True)
        else:
            g_trip = g_trip.reset_index(drop=True)

        used = set()

        if has_depart:
            for i in range(len(g_role)):
                target = float(g_role.loc[i, "expected_depart"])
                cand = g_trip[~g_trip.index.isin(used)].copy()
                if cand.empty:
                    break

                cand["delta"] = (cand["depart"] - target).abs()
                j = int(cand["delta"].idxmin())
                best_delta = float(cand.loc[j, "delta"])
                used.add(j)

                row = {**g_role.loc[i].to_dict(), **g_trip.loc[j, ["vehicle_id"]].to_dict()}
                for c in em_cols:
                    row[c] = g_trip.loc[j, c]
                if dist_col:
                    row[dist_col] = g_trip.loc[j, dist_col]
                if dur_col:
                    row[dur_col] = g_trip.loc[j, dur_col]
                row["matched_depart"] = g_trip.loc[j, "depart"]
                row["depart_delta"] = best_delta
                row["match_ok"] = (best_delta <= DEPART_TOL_SEC)
                matched_rows.append(row)

            if len(used) != len(g_role) or (len(g_role) != len(g_trip)):
                mismatch.append((pid, len(g_role), len(g_trip), "count_mismatch_or_partial_match"))
        else:
            # Fallback: match by row order if depart is missing
            m = min(len(g_role), len(g_trip))
            for i in range(m):
                row = {**g_role.loc[i].to_dict(), **g_trip.loc[i, ["vehicle_id"]].to_dict()}
                for c in em_cols:
                    row[c] = g_trip.loc[i, c]
                if dist_col:
                    row[dist_col] = g_trip.loc[i, dist_col]
                if dur_col:
                    row[dur_col] = g_trip.loc[i, dur_col]
                row["matched_depart"] = np.nan
                row["depart_delta"] = np.nan
                row["match_ok"] = False
                matched_rows.append(row)

            if len(g_role) != len(g_trip):
                mismatch.append((pid, len(g_role), len(g_trip), "no_depart_column_order_fallback"))

    df_match = pd.DataFrame(matched_rows)
    if df_match.empty:
        raise RuntimeError("No matches produced. Check vehicle ID format and resident filtering.")

    match_ok_rate = float(df_match["match_ok"].mean()) if "match_ok" in df_match.columns else np.nan
    logger.info("Matching OK (<= %ds): %.1f%%", DEPART_TOL_SEC, match_ok_rate * 100.0)

    # E) Carpool allocation and aggregation per person
    for c in em_cols:
        df_match[f"{c}_allocated"] = df_match[c] * df_match["factor"]

    agg = {c: "sum" for c in em_cols}
    agg.update({f"{c}_allocated": "sum" for c in em_cols})
    if dist_col:
        agg[dist_col] = "sum"
    if dur_col:
        agg[dur_col] = "sum"

    df_match["is_passenger"] = (df_match["role"] == "car_passenger").astype(int)
    agg["is_passenger"] = "mean"

    df_person = df_match.groupby("person_id", as_index=False).agg(agg)
    df_person = df_person.rename(columns={"is_passenger": "share_trips_as_passenger"})

    # Choose a main metric (prefer allocated CO2_abs)
    if "CO2_abs_allocated" in df_person.columns:
        main_metric = "CO2_abs_allocated"
    elif "CO2_allocated" in df_person.columns:
        main_metric = "CO2_allocated"
    else:
        main_metric = f"{em_cols[0]}_allocated"

    if dist_col:
        km = (df_person[dist_col] / 1000.0).replace(0, np.nan)
        df_person[f"{main_metric}_per_km"] = df_person[main_metric] / km

    # Summary stats
    metrics_for_summary = [f"{c}_allocated" for c in em_cols] + ([f"{main_metric}_per_km"] if dist_col else [])
    df_summary = pd.DataFrame(
        [{"metric": m, **summary_stats(df_person[m])} for m in metrics_for_summary]
    )[["metric", "mean", "median", "std", "p10", "p90", "min", "max"]]

    # Top emitters table
    d = df_person[["person_id", main_metric] + ([dist_col] if dist_col else [])].copy()
    d = d.sort_values(main_metric, ascending=False).reset_index(drop=True)
    total_main = float(d[main_metric].sum())
    d["share"] = d[main_metric] / total_main if total_main != 0 else np.nan
    d["cum_share"] = d["share"].cumsum()
    df_top = d.head(TOP_N).copy()

    g = gini(df_person[main_metric].to_numpy())

    # CSV outputs only
    df_match.to_csv(OUT_DIR / "matched_trips_with_roles.csv", index=False)
    df_person.to_csv(OUT_DIR / "emissions_by_person_carpool_adjusted.csv", index=False)
    df_summary.to_csv(OUT_DIR / "summary_stats_carpool_adjusted.csv", index=False)
    df_top.to_csv(OUT_DIR / f"top_{TOP_N}_persons_by_{main_metric}.csv", index=False)

    if mismatch:
        pd.DataFrame(
            mismatch,
            columns=["person_id", "n_car_trips_in_rou", "n_tripinfo_rows", "issue"]
        ).to_csv(OUT_DIR / "matching_issues.csv", index=False)

    logger.info("Main metric: %s | Gini: %.3f", main_metric, g)
    logger.info("Top10 share: %.3f%%", float(d.head(10)["share"].sum()) * 100.0)
    logger.info("Top20 share: %.3f%%", float(d.head(20)["share"].sum()) * 100.0)
    logger.info("Top%d share: %.3f%%", TOP_N, float(d.head(TOP_N)["share"].sum()) * 100.0)
    logger.info("CSV outputs written to: %s", OUT_DIR.resolve())


# Execute
run()

INFO | Residents (8 communes): 1647 persons
INFO | Car personTrips extracted from ROU: 4560
WARNING | tripinfo appears truncated -> repaired file written: ..\6-simulation\tripinfo.repaired.xml
INFO | tripinfo rows kept (private + residents): 4555
INFO | Matching OK (<= 120s): 97.3%
INFO | Main metric: CO2_abs_allocated | Gini: 0.460
INFO | Top10 share: 3.806%
INFO | Top20 share: 6.812%
INFO | Top50 share: 14.240%
INFO | CSV outputs written to: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_emissions_car_only


In [6]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Union
import contextlib
import io
import logging

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Visual diagnostics — Lorenz curve + distribution (Plotly, SVG export only)
# ============================================================

def plot_lorenz_and_distribution_svg(
    csv_path: Union[str, Path],
    output_svg: Union[str, Path],
    emission_col_candidates: Iterable[str] = ("CO2_abs_allocated", "CO2_allocated"),
    display_name: str = "CO₂",
    unit: str = "kg",
    convert_factor: float = 1e-6,
    bins: int = 40,
    filter_positive_hist: bool = True,
    include_zeros_lorenz: bool = True,
    width: int = 1200,
    height: int = 600,
    bar_color: str = "black",
    show: bool = True,
    quiet_export: bool = True,
) -> dict:
    
    csv_path = Path(csv_path)
    output_svg = Path(output_svg)

    df = pd.read_csv(csv_path)

    # -------------------------
    # Select metric column
    # -------------------------
    em_col = next((c for c in emission_col_candidates if c in df.columns), None)
    if em_col is None:
        raise ValueError(
            f"No emissions column found among {tuple(emission_col_candidates)}. "
            f"Available columns (excerpt): {df.columns.tolist()[:40]} ..."
        )

    # Clean and scale series
    s = pd.to_numeric(df[em_col], errors="coerce") * float(convert_factor)
    s = s.replace([np.inf, -np.inf], np.nan).dropna()

    # -------------------------
    # Histogram (right panel)
    # -------------------------
    s_hist = s.copy()
    if filter_positive_hist:
        s_hist = s_hist[s_hist > 0]
    if s_hist.empty:
        raise ValueError("No data available for histogram after filtering.")

    mean_val = float(s_hist.mean())
    counts, edges = np.histogram(s_hist.to_numpy(), bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = edges[1:] - edges[:-1]

    # -------------------------
    # Lorenz + Gini (left panel)
    # -------------------------
    s_lor = s.copy()
    s_lor = s_lor[s_lor >= 0] if include_zeros_lorenz else s_lor[s_lor > 0]

    if s_lor.empty or float(s_lor.sum()) <= 0:
        raise ValueError("Cannot compute Lorenz curve (empty data or zero total).")

    x = np.sort(s_lor.to_numpy())
    total = float(x.sum())
    cum = np.cumsum(x) / total
    pop = np.arange(1, len(x) + 1) / len(x)

    pop_l = np.concatenate(([0.0], pop))
    cum_l = np.concatenate(([0.0], cum))

    area = np.trapz(cum_l, pop_l)
    gini = 1.0 - 2.0 * float(area)

    pop_pct = pop_l * 100
    cum_pct = cum_l * 100

    # -------------------------
    # Figure
    # -------------------------
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Lorenz Curve — Gini = {gini:.3f}",
            f"{display_name} distribution (per person)"
        ),
        horizontal_spacing=0.10
    )

    # Lorenz curve
    fig.add_trace(
        go.Scatter(
            x=pop_pct, y=cum_pct, mode="lines", name="Lorenz",
            line=dict(color="black", width=3)
        ),
        row=1, col=1
    )

    # Equality line
    fig.add_trace(
        go.Scatter(
            x=[0, 100], y=[0, 100], mode="lines", name="Equality",
            line=dict(color="red", width=2, dash="dash")
        ),
        row=1, col=1
    )

    # Histogram
    fig.add_trace(
        go.Bar(
            x=centers,
            y=counts,
            width=widths,
            marker=dict(color=bar_color, line=dict(color="black", width=0.8)),
            hovertemplate=f"{display_name}: %{{x:.4g}} {unit}<br>People: %{{y}}<extra></extra>",
            showlegend=False,
        ),
        row=1, col=2
    )

    # Mean line + annotation
    fig.add_vline(x=mean_val, line_width=3, line_color="red", row=1, col=2)
    fig.add_annotation(
        x=mean_val, y=0.85, xref="x2", yref="paper",
        text=f"Mean = {mean_val:.3g} {unit}",
        showarrow=True, arrowhead=0, ax=40, ay=-30,
        font=dict(color="red", size=16),
        xanchor="left", yanchor="top",
        bgcolor="white", bordercolor="red", borderwidth=1, borderpad=3,
    )

    fig.update_layout(
        template="simple_white",
        width=width,
        height=height,
        margin=dict(l=70, r=25, t=80, b=70),
        font=dict(size=18, color="black"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.02),
    )

    fig.update_xaxes(
        title_text="Cumulative share of people (%)", range=[0, 100],
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
        ticksuffix="%",
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Cumulative share of emissions (%)", range=[0, 102],
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
        ticksuffix="%",
        row=1, col=1
    )

    fig.update_xaxes(
        title_text=f"{display_name} ({unit})",
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Number of people",
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
        row=1, col=2
    )

    if show:
        fig.show()

    # -------------------------
    # SVG export (only)
    # -------------------------
    output_svg.parent.mkdir(parents=True, exist_ok=True)

    if quiet_export:
        # Silence Kaleido/Choreographer (both logging + stdout/stderr)
        for name in ("kaleido", "choreographer"):
            logging.getLogger(name).setLevel(logging.ERROR)

        root = logging.getLogger()
        root.setLevel(logging.WARNING)
        for h in root.handlers:
            h.setLevel(logging.WARNING)

        _buf = io.StringIO()
        with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):
            fig.write_image(output_svg)
    else:
        fig.write_image(output_svg)

    return {"metric": em_col, "gini": float(gini), "mean": float(mean_val), "n": int(len(s_lor))}


# ============================================================
# Example call (SVG only)
# ============================================================
stats = plot_lorenz_and_distribution_svg(
    csv_path="outputs_emissions_car_only/emissions_by_person_carpool_adjusted.csv",
    output_svg="outputs_emissions_car_only/lorenz_distribution_CO2.svg",
    display_name="CO₂",
    unit="kg",
    convert_factor=1e-6,
    bins=40,
    show=True,
    quiet_export=True,
)

print(stats)

C:\Users\ngale\AppData\Local\Temp\ipykernel_18236\348183062.py:86: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



{'metric': 'CO2_abs_allocated', 'gini': 0.4601546949955343, 'mean': 6.1648085329014215, 'n': 1374}


In [8]:
stats = plot_lorenz_and_distribution_svg(
    csv_path="outputs_emissions_car_only/emissions_by_person_carpool_adjusted.csv",
    output_svg="outputs_emissions_car_only/lorenz_distribution_PMx.svg",
    emission_col_candidates=("PMx_abs_allocated",),
    display_name="PMx",
    unit="g",
    convert_factor=1e-3,  # e.g., mg -> g
    bins=40,
    width=1200,
    height=600,
    show=True,
    quiet_export=True,
)

print(stats)

C:\Users\ngale\AppData\Local\Temp\ipykernel_18236\348183062.py:86: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



{'metric': 'PMx_abs_allocated', 'gini': 0.5498708061731314, 'mean': 1.020021657573508, 'n': 1374}


2. Diagnostics — Top 50 CO2 vs all

In [9]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

# =============================================================================
# Top 50 CO2 vs all (distance + fleet mix enrichment, CSV only)
# =============================================================================

# -----------------------------
# PATHS
# -----------------------------
EM_CSV = Path("outputs_emissions_car_only/emissions_by_person_carpool_adjusted.csv")
TOP_CSV = Path("outputs_emissions_car_only/top_50_persons_by_CO2_abs_allocated.csv")
MAP_CSV = Path("../5-vehicules/person_to_emissionClass.csv")

OUT_DIR = Path("outputs_top50_diagnosis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# LOAD
# -----------------------------
df_em = pd.read_csv(EM_CSV)
df_top = pd.read_csv(TOP_CSV)
df_map = pd.read_csv(MAP_CSV)

# Normalise person_id type (prefer string for robust joins)
df_em["person_id"] = df_em["person_id"].astype(str)
df_top["person_id"] = df_top["person_id"].astype(str)
df_map["person_id"] = df_map["person_id"].astype(str)

# Restrict mapping to persons present in df_em
df_map = df_map[df_map["person_id"].isin(df_em["person_id"])].copy()

# -----------------------------
# Parse emissionClass -> (vehcat, fuel, euro)
# -----------------------------
def parse_emission_class(emission_class: str):
    """
    Parse an HBEFA4 emissionClass string like:
      HBEFA4/PC_diesel_Euro-4
    and return:
      (vehcat, fuel, euro)
    """
    if pd.isna(emission_class):
        return ("unknown", "unknown", "unknown")

    s = str(emission_class)
    tail = s.split("/")[-1]   # e.g., PC_diesel_Euro-4
    parts = tail.split("_")

    vehcat = parts[0] if len(parts) > 0 else "unknown"
    fuel = parts[1] if len(parts) > 1 else "unknown"
    euro = "_".join(parts[2:]) if len(parts) > 2 else "unknown"
    return vehcat, fuel, euro

vehcat_fuel_euro = df_map["emissionClass"].apply(parse_emission_class)
df_map[["vehcat", "fuel", "euro"]] = pd.DataFrame(vehcat_fuel_euro.tolist(), index=df_map.index)

# -----------------------------
# Merge + derived metrics
# -----------------------------
df = df_em.merge(df_map[["person_id", "emissionClass", "fuel", "euro"]], on="person_id", how="left")

# Distance (km)
if "routeLength" in df.columns:
    df["distance_km"] = pd.to_numeric(df["routeLength"], errors="coerce") / 1000.0
else:
    df["distance_km"] = np.nan

# Unit conversions (adapt if your tripinfo units differ)
# Here we keep your prior assumption: CO2_abs_allocated in mg -> kg, PMx_abs_allocated in mg -> g
if "CO2_abs_allocated" in df.columns:
    df["CO2_kg"] = pd.to_numeric(df["CO2_abs_allocated"], errors="coerce") * 1e-6
else:
    df["CO2_kg"] = np.nan

if "PMx_abs_allocated" in df.columns:
    df["PMx_g"] = pd.to_numeric(df["PMx_abs_allocated"], errors="coerce") * 1e-3
else:
    df["PMx_g"] = np.nan

# CO2 intensity if available (mg/km -> g/km)
if "CO2_abs_allocated_per_km" in df.columns:
    df["CO2_g_per_km"] = pd.to_numeric(df["CO2_abs_allocated_per_km"], errors="coerce") * 1e-3

# Top50 subset
top_ids = set(df_top["person_id"])
df_top50 = df[df["person_id"].isin(top_ids)].copy()

# -----------------------------
# Helpers
# -----------------------------
def stats_block(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return pd.Series({"n": 0, "mean": np.nan, "median": np.nan, "p10": np.nan, "p90": np.nan})
    return pd.Series({
        "n": int(s.shape[0]),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "p10": float(np.percentile(s, 10)),
        "p90": float(np.percentile(s, 90)),
    })

def mix_table(df_all: pd.DataFrame, df_top: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Build a composition/enrichment table for a categorical attribute.
    enrichment = share_top / share_all
    coverage   = count_top / count_all
    """
    all_counts = df_all[col].fillna("unknown").value_counts()
    top_counts = df_top[col].fillna("unknown").value_counts()

    tbl = pd.concat(
        [all_counts.rename("count_all"), top_counts.rename("count_top")],
        axis=1
    ).fillna(0)

    tbl["share_all"] = tbl["count_all"] / tbl["count_all"].sum() if tbl["count_all"].sum() > 0 else np.nan
    tbl["share_top"] = tbl["count_top"] / tbl["count_top"].sum() if tbl["count_top"].sum() > 0 else np.nan
    tbl["enrichment"] = np.where(tbl["share_all"] > 0, tbl["share_top"] / tbl["share_all"], np.nan)
    tbl["coverage"] = np.where(tbl["count_all"] > 0, tbl["count_top"] / tbl["count_all"], np.nan)
    return tbl.sort_values("enrichment", ascending=False)

# =============================================================================
# 1) Distance diagnostics (all vs top50)
# =============================================================================
dist_all = stats_block(df["distance_km"])
dist_top = stats_block(df_top50["distance_km"])

dist_tbl = pd.DataFrame({"all": dist_all, "top50": dist_top}).T
dist_tbl.to_csv(OUT_DIR / "distance_stats_all_vs_top50.csv", index=True)

p90_all = float(np.percentile(df["distance_km"].replace([np.inf, -np.inf], np.nan).dropna(), 90)) if df["distance_km"].notna().any() else np.nan
share_top_above_p90 = float((df_top50["distance_km"] > p90_all).mean()) if np.isfinite(p90_all) else np.nan

total_co2 = float(df["CO2_kg"].replace([np.inf, -np.inf], np.nan).dropna().sum())
top50_co2 = float(df_top50["CO2_kg"].replace([np.inf, -np.inf], np.nan).dropna().sum())
share_co2_top50 = float(top50_co2 / total_co2) if total_co2 > 0 else np.nan

# =============================================================================
# 2) Fleet mix enrichment (fuel + euro)
# =============================================================================
fuel_tbl = mix_table(df, df_top50, "fuel")
fuel_tbl.to_csv(OUT_DIR / "fuel_mix_enrichment_top50_vs_all.csv")

euro_tbl = mix_table(df, df_top50, "euro")
euro_tbl.to_csv(OUT_DIR / "euro_mix_enrichment_top50_vs_all.csv")

# Focus metric: "old diesel" (Euro-0..Euro-3)
old_euros = {"Euro-0", "Euro-1", "Euro-2", "Euro-3"}
df["is_old_diesel"] = ((df["fuel"] == "diesel") & (df["euro"].isin(old_euros))).astype(int)
df_top50["is_old_diesel"] = ((df_top50["fuel"] == "diesel") & (df_top50["euro"].isin(old_euros))).astype(int)

old_diesel_all = float(df["is_old_diesel"].mean()) if len(df) else np.nan
old_diesel_top = float(df_top50["is_old_diesel"].mean()) if len(df_top50) else np.nan

# =============================================================================
# 3) Compact summary (single CSV for reporting)
# =============================================================================
summary = pd.DataFrame([{
    "n_all_persons": int(df["person_id"].nunique()),
    "n_top50_persons": int(df_top50["person_id"].nunique()),
    "p90_distance_all_km": p90_all,
    "share_top50_above_p90_distance": share_top_above_p90,
    "share_co2_top50": share_co2_top50,
    "old_diesel_share_all": old_diesel_all,
    "old_diesel_share_top50": old_diesel_top,
}])

summary.to_csv(OUT_DIR / "diagnostics_summary.csv", index=False)

# Console summary (minimal)
print("[INFO] Distance stats (all vs top50) written:", (OUT_DIR / "distance_stats_all_vs_top50.csv").resolve())
print("[INFO] Fuel enrichment table written:", (OUT_DIR / "fuel_mix_enrichment_top50_vs_all.csv").resolve())
print("[INFO] Euro enrichment table written:", (OUT_DIR / "euro_mix_enrichment_top50_vs_all.csv").resolve())
print("[INFO] Summary written:", (OUT_DIR / "diagnostics_summary.csv").resolve())
print(f"[INFO] Top50 CO2 share: {share_co2_top50:.1%}" if pd.notna(share_co2_top50) else "[INFO] Top50 CO2 share: n/a")
print(f"[INFO] Top50 above P90 distance: {share_top_above_p90:.1%}" if pd.notna(share_top_above_p90) else "[INFO] Top50 above P90 distance: n/a")
print(f"[INFO] Old diesel share: all={old_diesel_all:.1%} vs top50={old_diesel_top:.1%}" if pd.notna(old_diesel_all) else "[INFO] Old diesel share: n/a")
print(f"[OK] CSV outputs in: {OUT_DIR.resolve()}")

[INFO] Distance stats (all vs top50) written: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis\distance_stats_all_vs_top50.csv
[INFO] Fuel enrichment table written: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis\fuel_mix_enrichment_top50_vs_all.csv
[INFO] Euro enrichment table written: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis\euro_mix_enrichment_top50_vs_all.csv
[INFO] Summary written: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis\diagnostics_summary.csv
[INFO] Top50 CO2 share: 14.2%
[INFO] Top50 above P90 distance: 88.0%
[INFO] Old diesel share: all=9.2% vs top50=22.0%
[OK] CSV outputs in: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Union
import contextlib
import io
import logging

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# Visual diagnostics — Distance histogram + distance-vs-CO2 scatter (SVG only)
# =============================================================================
#
# Purpose
# -------
# Create a two-panel diagnostic figure comparing:
#   (1) Total daily distance distribution (all residents vs top-50 CO2 emitters)
#   (2) Distance vs allocated CO2 (scatter, top-50 highlighted)
#
# Output
# ------
# - SVG only (requires Kaleido). No PNG/HTML export.
#
# Notes
# -----
# - When quiet_export=True, Kaleido/Choreographer verbose logs are silenced during export.
# =============================================================================


def plot_distance_hist_and_scatter_all_vs_top50_svg(
    emissions_by_person_csv: Union[str, Path],
    top50_csv: Union[str, Path],
    output_svg: Union[str, Path],
    person_id_col: str = "person_id",
    distance_col: str = "routeLength",
    co2_col: str = "CO2_abs_allocated",
    distance_factor: float = 1 / 1000.0,   # m -> km
    co2_factor: float = 1e-6,              # mg -> kg
    bins: int = 40,
    width: int = 1500,
    height: int = 650,
    show: bool = True,
    quiet_export: bool = True,
) -> dict:
    """
    Generate the diagnostic Plotly figure and export to SVG.

    Parameters
    ----------
    emissions_by_person_csv:
        CSV containing per-person aggregated metrics.
    top50_csv:
        CSV containing a column with top-50 person IDs (person_id_col).
    output_svg:
        Destination SVG path.
    distance_factor, co2_factor:
        Unit conversions applied to distance_col and co2_col.
    show:
        If True, display the figure in the notebook.
    quiet_export:
        If True, suppress Kaleido logs and stdout/stderr during SVG export.

    Returns
    -------
    dict:
        {"n_all": int, "n_top50": int, "mean_distance_all_km": float}
    """
    emissions_by_person_csv = Path(emissions_by_person_csv)
    top50_csv = Path(top50_csv)
    output_svg = Path(output_svg)

    df_all = pd.read_csv(emissions_by_person_csv)
    df_top = pd.read_csv(top50_csv)

    # Validate required columns
    for col in (person_id_col, distance_col, co2_col):
        if col not in df_all.columns:
            raise KeyError(f"Column '{col}' not found in {emissions_by_person_csv}.")
    if person_id_col not in df_top.columns:
        raise KeyError(f"Column '{person_id_col}' not found in {top50_csv}.")

    # Normalise IDs for matching
    df_all[person_id_col] = df_all[person_id_col].astype(str)
    df_top[person_id_col] = df_top[person_id_col].astype(str)
    top_ids = set(df_top[person_id_col].dropna().unique())

    # Conversions (distance + CO2)
    dist_all_km = pd.to_numeric(df_all[distance_col], errors="coerce") * float(distance_factor)
    co2_all_kg = pd.to_numeric(df_all[co2_col], errors="coerce") * float(co2_factor)

    mask_top = df_all[person_id_col].isin(top_ids)
    dist_top_km = dist_all_km[mask_top].replace([np.inf, -np.inf], np.nan).dropna()
    dist_all_km = dist_all_km.replace([np.inf, -np.inf], np.nan).dropna()

    mean_all = float(dist_all_km.mean()) if len(dist_all_km) else np.nan

    # Scatter dataset (plot-ready)
    df_sc = df_all[[person_id_col, distance_col, co2_col]].copy()
    df_sc["dist_km"] = pd.to_numeric(df_sc[distance_col], errors="coerce") * float(distance_factor)
    df_sc["co2_kg"] = pd.to_numeric(df_sc[co2_col], errors="coerce") * float(co2_factor)
    df_sc = df_sc.replace([np.inf, -np.inf], np.nan).dropna(subset=["dist_km", "co2_kg"])
    df_sc["is_top50"] = df_sc[person_id_col].isin(top_ids)

    df_sc_others = df_sc[~df_sc["is_top50"]]
    df_sc_top = df_sc[df_sc["is_top50"]]

    # Colors (keep consistent across the repo)
    color_all = "#3274A1"
    color_top = "#E1812C"
    color_scat = "#8CBF6A"
    color_stop = "#C44E52"

    # Figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Total daily distance (km) — All vs Top-50 CO₂",
            "Distance vs CO₂ — All vs Top-50",
        ),
        horizontal_spacing=0.10,
    )

    # Left: histogram overlay
    fig.add_trace(
        go.Histogram(
            x=dist_all_km, nbinsx=bins,
            name="All residents",
            marker_color=color_all,
            opacity=0.65,
            marker=dict(line=dict(color="black", width=0.5)),
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Histogram(
            x=dist_top_km, nbinsx=bins,
            name="Top-50 CO₂",
            marker_color=color_top,
            opacity=0.90,
            marker=dict(line=dict(color="black", width=0.8)),
        ),
        row=1, col=1,
    )

    # Mean line (all)
    if np.isfinite(mean_all):
        fig.add_vline(
            x=mean_all, line_width=3, line_dash="dash",
            line_color="red", row=1, col=1,
        )
        fig.add_annotation(
            x=mean_all, y=0.85, xref="x1", yref="paper",
            text=f"Mean = {mean_all:.1f} km",
            showarrow=True, arrowhead=0, ax=50, ay=-30,
            font=dict(color="red", size=16),
            xanchor="left", yanchor="top",
            bgcolor="white",
            bordercolor="red",
            borderwidth=1,
            borderpad=3,
        )

    # Right: scatter
    fig.add_trace(
        go.Scatter(
            x=df_sc_others["dist_km"], y=df_sc_others["co2_kg"],
            mode="markers",
            name="All residents",
            marker=dict(size=6, color=color_scat, line=dict(width=0.3, color="gray")),
            opacity=0.50,
            hovertemplate="Distance = %{x:.1f} km<br>CO₂ = %{y:.3g} kg<extra></extra>",
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=df_sc_top["dist_km"], y=df_sc_top["co2_kg"],
            mode="markers",
            name="Top-50 CO₂",
            marker=dict(size=10, color=color_stop, line=dict(width=0.5, color="black")),
            opacity=0.95,
            hovertemplate="TOP-50<br>Distance = %{x:.1f} km<br>CO₂ = %{y:.3g} kg<extra></extra>",
        ),
        row=1, col=2,
    )

    # Layout
    fig.update_layout(
        template="simple_white",
        width=width,
        height=height,
        barmode="overlay",
        margin=dict(l=70, r=25, t=80, b=70),
        font=dict(size=18, color="black"),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            font=dict(size=15),
        ),
    )

    fig.update_xaxes(
        title_text="Total distance (km)", row=1, col=1,
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )
    fig.update_yaxes(
        title_text="Number of people", row=1, col=1,
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )

    fig.update_xaxes(
        title_text="Total distance (km)", row=1, col=2,
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )
    fig.update_yaxes(
        title_text="Allocated CO₂ (kg)", row=1, col=2,
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )

    if show:
        fig.show()

    # SVG export only (optionally silent)
    output_svg.parent.mkdir(parents=True, exist_ok=True)

    if quiet_export:
        for name in ("kaleido", "choreographer"):
            logging.getLogger(name).setLevel(logging.ERROR)

        root = logging.getLogger()
        root.setLevel(logging.WARNING)
        for h in root.handlers:
            h.setLevel(logging.WARNING)

        _buf = io.StringIO()
        with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):
            fig.write_image(output_svg)
    else:
        fig.write_image(output_svg)

    return {
        "n_all": int(df_all[person_id_col].nunique()),
        "n_top50": int(len(top_ids)),
        "mean_distance_all_km": float(mean_all),
        "output_svg": str(output_svg),
    }



stats = plot_distance_hist_and_scatter_all_vs_top50_svg(
    emissions_by_person_csv="outputs_emissions_car_only/emissions_by_person_carpool_adjusted.csv",
    top50_csv="outputs_emissions_car_only/top_50_persons_by_CO2_abs_allocated.csv",
    output_svg="outputs_top50_diagnosis/dist_hist_scatter_all_vs_top50.svg",
    distance_col="routeLength",
    co2_col="CO2_abs_allocated",
    distance_factor=1 / 1000.0,
    co2_factor=1e-6,
    bins=40,
    width=1500,
    height=650,
    show=True,
    quiet_export=True,
)

print(stats)

{'n_all': 1374, 'n_top50': 50, 'mean_distance_all_km': 31.449477452692868, 'output_svg': 'outputs_top50_diagnosis\\dist_hist_scatter_all_vs_top50.svg'}


In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go


def read_csv_flexible(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")


def fuel_from_emission_class(ec: str) -> str:
    if pd.isna(ec):
        return "Unknown"
    s = str(ec).lower()
    if "bev" in s:
        return "BEV (electric)"
    if "phev" in s:
        return "PHEV"
    if "lpg" in s:
        return "LPG"
    if "diesel" in s:
        return "Diesel"
    if "petrol" in s:
        return "Petrol"
    return "Other"


def pretty_emission_class(ec: str) -> str:
    if pd.isna(ec):
        return "Unknown"
    s = str(ec)
    s = s.replace("HBEFA4/", "").replace("PC_", "")
    s = s.replace("_Euro-", " Euro-").replace("_", " ")
    return s.strip()


def plot_motorisation_bars_only_with_ratio(
    emissions_by_person_csv: str | Path,
    top50_csv: str | Path,
    person_to_emissionClass_csv: str | Path,
    group_level: str = "fuel",
    person_id_col: str = "person_id",
    emission_class_col: str = "emissionClass",
    width: int = 1000,
    height: int = 650,
    output_svg: str | Path | None = None,
):
    df_all = read_csv_flexible(emissions_by_person_csv)
    df_top = read_csv_flexible(top50_csv)
    df_map = read_csv_flexible(person_to_emissionClass_csv)

    # IDs
    df_all[person_id_col] = df_all[person_id_col].astype(str)
    df_top[person_id_col] = df_top[person_id_col].astype(str)
    df_map[person_id_col] = df_map[person_id_col].astype(str)

    # Filter mapping to fleet of interest
    fleet_ids = set(df_all[person_id_col].unique())
    df_map_f = df_map[df_map[person_id_col].isin(fleet_ids)].copy()
    df_map_f = df_map_f.drop_duplicates(subset=[person_id_col], keep="first")

    # Labels
    df_map_f["fuel_type"] = df_map_f[emission_class_col].apply(fuel_from_emission_class)
    df_map_f["emissionClass_pretty"] = df_map_f[emission_class_col].apply(pretty_emission_class)

    if group_level == "fuel":
        motor_col = "fuel_type"
        title = "Powertrain distribution — All fleet vs Top-50 CO₂"
    elif group_level == "emissionClass":
        motor_col = "emissionClass_pretty"
        title = "Emission class distribution — All fleet vs Top-50 CO₂"
    else:
        raise ValueError("group_level must be 'fuel' or 'emissionClass'")

    # Merge fleet + powertrain
    df_fleet = df_all[[person_id_col]].merge(
        df_map_f[[person_id_col, motor_col]],
        on=person_id_col,
        how="left",
    )
    df_fleet[motor_col] = df_fleet[motor_col].fillna("Unknown")

    top_ids = set(df_top[person_id_col].unique())
    df_fleet["is_top50"] = df_fleet[person_id_col].isin(top_ids)

    N_all = int(df_fleet[person_id_col].nunique())
    N_top = int(df_fleet.loc[df_fleet["is_top50"], person_id_col].nunique())

    # Counts
    all_counts = df_fleet[motor_col].value_counts(dropna=False)
    top_counts = df_fleet.loc[df_fleet["is_top50"], motor_col].value_counts(dropna=False)

    summary = (
        pd.DataFrame({"count_all": all_counts, "count_top50": top_counts})
        .fillna(0)
        .astype({"count_all": int, "count_top50": int})
        .reset_index()
    )
    summary = summary.rename(columns={summary.columns[0]: "motorisation"})

    summary["share_all"] = summary["count_all"] / max(N_all, 1)
    summary["share_top50"] = summary["count_top50"] / max(N_top, 1)

    # Over-representation ratio (Top-50 vs fleet)
    summary["ratio"] = np.where(
        summary["share_all"] > 0,
        summary["share_top50"] / summary["share_all"],
        np.nan,
    )

    # Sort: dominant in top-50 first
    summary = summary.sort_values(
        by=["count_top50", "count_all", "motorisation"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    # ── Colors ──
    color_all = "#3274A1"
    color_top = "#E1812C"

    # ── Figure ──
    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            y=summary["motorisation"],
            x=summary["share_all"] * 100,
            orientation="h",
            name="All fleet",
            marker_color=color_all,
            marker=dict(line=dict(color="black", width=0.5)),
            hovertemplate="%{y}<br>Fleet: %{x:.1f}%<extra></extra>",
        )
    )

    fig.add_trace(
        go.Bar(
            y=summary["motorisation"],
            x=summary["share_top50"] * 100,
            orientation="h",
            name="Top-50 CO₂",
            marker_color=color_top,
            marker=dict(line=dict(color="black", width=0.5)),
            hovertemplate="%{y}<br>Top-50: %{x:.1f}%<extra></extra>",
        )
    )

    # ── Ratio legend box (top-right annotation) ──
    ratio_lines = ["<b>Ratio Top-50 / Fleet</b>"]
    for _, row in summary.iterrows():
        r = row["ratio"]
        if pd.notna(r) and np.isfinite(r) and row["count_top50"] > 0:
            ratio_lines.append(f"{row['motorisation']}: {r:.2f}")

    ratio_text = "<br>".join(ratio_lines)

    fig.add_annotation(
        x=1.0, y=1.0,
        xref="paper", yref="paper",
        xanchor="right", yanchor="top",
        text=ratio_text,
        showarrow=False,
        font=dict(size=13, color="black", family="Courier New"),
        align="left",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1.5,
        borderpad=8,
    )

    # ── Layout ──
    xmax = float(max(summary["share_all"].max(), summary["share_top50"].max()) * 100)
    fig.update_xaxes(range=[0, min(100, xmax + 5)])

    fig.update_layout(
        template="simple_white",
        title=dict(text=title, font=dict(size=20)),
        width=width,
        height=height,
        barmode="group",
        margin=dict(l=120, r=25, t=80, b=70),
        font=dict(size=18, color="black"),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02,
            xanchor="left", x=0.02,
            font=dict(size=15),
        ),
    )

    fig.update_xaxes(
        title_text="Share (%)",
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )
    fig.update_yaxes(
        title_text="Powertrain",
        showline=True, linewidth=2, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2, ticklen=7, zeroline=False,
    )

    fig.show()
    
    if output_svg is not None:
        output_svg = Path(output_svg)
        fig.write_image(output_svg)
        print(f"[OK] SVG saved: {output_svg.resolve()}")

    return {"N_all": N_all, "N_top50": N_top}


# %% ── Direct call  ──────────────────────────
plot_motorisation_bars_only_with_ratio(
    emissions_by_person_csv="outputs_emissions_car_only/emissions_by_person_carpool_adjusted.csv",
    top50_csv="outputs_emissions_car_only/top_50_persons_by_CO2_abs_allocated.csv",
    person_to_emissionClass_csv="../5-vehicules/person_to_emissionClass.csv",
    group_level="fuel",
    output_svg="outputs_top50_diagnosis/motorisation_fuel.svg",
)

[OK] SVG saved: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_top50_diagnosis\motorisation_fuel.svg


{'N_all': 1374, 'N_top50': 50}

In [16]:
from __future__ import annotations

from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# =============================================================================
# Emissions totals — SUMO tripinfo aggregation (CSV output)
# =============================================================================
#
# Purpose
# -------
# Aggregate total emissions directly from a SUMO tripinfo.xml file by summing all
# attributes found in <emissions .../> sub-elements.
#
# Outputs
# -------
# - A single CSV containing raw totals and converted totals (unit conversions are configurable).
#
# Notes
# -----
# - tripinfo.xml can sometimes be truncated; we repair it if the closing </tripinfos> tag is missing.
# - By default, "vaporized" trips are excluded.
# =============================================================================

# -----------------------------
# CONFIG
# -----------------------------
TRIPINFO_XML = Path("../6-simulation/tripinfo.xml")  # adjust if needed
OUT_CSV = Path("outputs_emissions_car_only/totals_emissions_tripinfo.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

SKIP_VAPORIZED = True

# Unit conversions aligned with your assumptions:
# - pollutants (abs): mg -> kg or g
# - fuel_abs: assumed mg fuel -> kg
# - electricity_abs: Wh -> kWh
CONV = {
    "CO2_abs":         ("mg", "kg", 1e-6),
    "CO_abs":          ("mg", "g",  1e-3),
    "HC_abs":          ("mg", "g",  1e-3),
    "NOx_abs":         ("mg", "g",  1e-3),
    "PMx_abs":         ("mg", "g",  1e-3),
    "fuel_abs":        ("mg", "kg", 1e-6),
    "electricity_abs": ("Wh", "kWh", 1e-3),
}


# -----------------------------
# Helpers
# -----------------------------
def repair_tripinfo_if_needed(path: Path) -> Path:
    """
    Repair tripinfo.xml if the closing </tripinfos> tag is missing.
    ElementTree may fail on truncated XML.
    """
    if not path.exists():
        raise FileNotFoundError(f"tripinfo file not found: {path.resolve()}")

    data = path.read_bytes()
    if b"</tripinfos>" in data[-50000:]:
        return path

    if b"<tripinfos" not in data[:200000]:
        raise RuntimeError("'<tripinfos ...>' not found: this file does not look like a SUMO tripinfo output.")

    repaired = path.with_suffix(".repaired.xml")
    repaired.write_bytes(data + b"\n</tripinfos>\n")
    print(f"[WARN] tripinfo appears truncated -> repaired file written: {repaired}")
    return repaired


def compute_total_emissions(tripinfo_path: Path, skip_vaporized: bool = True):
    """
    Sum all numeric attributes found in <emissions .../> for every <tripinfo> element.
    Returns:
      totals (dict), n_tripinfo, n_with_emissions, n_skipped_vaporized
    """
    totals: dict[str, float] = {}
    n_tripinfo = 0
    n_with_em = 0
    n_skipped_vap = 0

    for _event, elem in ET.iterparse(tripinfo_path, events=("end",)):
        if elem.tag != "tripinfo":
            continue

        n_tripinfo += 1

        if skip_vaporized:
            vap = (elem.get("vaporized") or "").strip().lower()
            if vap in {"true", "1", "yes"}:
                n_skipped_vap += 1
                elem.clear()
                continue

        em = elem.find("emissions")
        if em is not None:
            n_with_em += 1
            for k, v in em.attrib.items():
                try:
                    totals[k] = totals.get(k, 0.0) + float(v)
                except Exception:
                    # Skip non-numeric values
                    pass

        elem.clear()

    return totals, n_tripinfo, n_with_em, n_skipped_vap


# -----------------------------
# RUN
# -----------------------------
fixed_path = repair_tripinfo_if_needed(TRIPINFO_XML)
totals, n_tripinfo, n_with_em, n_skipped_vap = compute_total_emissions(fixed_path, skip_vaporized=SKIP_VAPORIZED)

rows = []
for k, total_raw in totals.items():
    raw_unit, conv_unit, factor = CONV.get(k, ("raw", "raw", 1.0))
    rows.append(
        {
            "emission": k,
            "total_raw": total_raw,
            "raw_unit": raw_unit,
            "factor_to_converted": factor,
            "total_converted": total_raw * factor,
            "converted_unit": conv_unit,
        }
    )

df_totals = (
    pd.DataFrame(rows)
    .sort_values("total_converted", ascending=False)
    .reset_index(drop=True)
)

print(
    f"[INFO] tripinfo elements: {n_tripinfo:,} | "
    f"with <emissions>: {n_with_em:,} | "
    f"vaporized skipped: {n_skipped_vap:,}"
)

df_totals.to_csv(OUT_CSV, index=False)
print(f"[OK] CSV written: {OUT_CSV.resolve()}")

df_totals.head(20)

[WARN] tripinfo appears truncated -> repaired file written: ..\6-simulation\tripinfo.repaired.xml
[INFO] tripinfo elements: 42,720 | with <emissions>: 42,720 | vaporized skipped: 0
[OK] CSV written: C:\Users\ngale\Downloads\SimulationV2\SUMO\7-output_analysis\outputs_emissions_car_only\totals_emissions_tripinfo.csv


,emission,total_raw,raw_unit,factor_to_converted,total_converted,converted_unit
0,NOx_abs,5.193339e+08,mg,0.001000,519333.872063,g
1,CO_abs,2.587427e+08,mg,0.001000,258742.687368,g
2,CO2_abs,1.033656e+11,mg,0.000001,103365.562223,kg
3,fuel_abs,3.308142e+10,mg,0.000001,33081.423486,kg
4,HC_abs,2.904570e+07,mg,0.001000,29045.696800,g
5,PMx_abs,1.920506e+07,mg,0.001000,19205.055060,g
6,electricity_abs,5.396639e+05,Wh,0.001000,539.663910,kWh


In [17]:
import pandas as pd

# =============================================================================
# Sanity check — Compare simulated daily CO2 to a reference inventory (road transport)
# =============================================================================
#
# Purpose
# -------
# Compute simple per-capita/per-agent CO2 indicators to compare:
#   - Simulation (SUMO tripinfo total CO2 per day)
#   - Reference inventory (annual road-transport CO2 converted to daily)
#
# Notes
# -----
# - This is a high-level plausibility check, not a calibration.
# - Ensure the simulation day definition (time window, agents included) is comparable.
# =============================================================================

# -----------------------------
# INPUTS (edit these values)
# -----------------------------
SIM_CO2_T_PER_DAY = 103.365562223   # total CO2 from simulation (tonnes CO2 / day)
N_AGENTS = 17107                    # number of simulated agents (persons)

# Reference (inventory)
REF_ROAD_T_PER_YEAR = 283_417       # road transport emissions (tCO2e / year)
REF_TOTAL_T_PER_YEAR = 1_883_617    # total emissions used to derive implicit population (tCO2e / year)
REF_T_PER_CAPITA_PER_YEAR = 11.6    # tCO2e / capita / year (used in the inventory)

# -----------------------------
# DERIVED REFERENCE VALUES
# -----------------------------
# Implicit population behind the inventory ratio (total / per-capita)
REF_POP_IMPLIED = REF_TOTAL_T_PER_YEAR / REF_T_PER_CAPITA_PER_YEAR

# Convert annual road emissions to daily
REF_ROAD_T_PER_DAY = REF_ROAD_T_PER_YEAR / 365.0

# -----------------------------
# INDICATORS
# -----------------------------
SIM_KG_PER_AGENT_DAY = SIM_CO2_T_PER_DAY * 1000.0 / N_AGENTS
REF_KG_PER_CAPITA_DAY = REF_ROAD_T_PER_DAY * 1000.0 / REF_POP_IMPLIED

RATIO_SIM_OVER_REF = SIM_KG_PER_AGENT_DAY / REF_KG_PER_CAPITA_DAY if REF_KG_PER_CAPITA_DAY > 0 else float("nan")

# Extrapolate simulation total to the implied reference population
SCALED_SIM_T_PER_DAY = SIM_CO2_T_PER_DAY * (REF_POP_IMPLIED / N_AGENTS)

# -----------------------------
# REPORT TABLE
# -----------------------------
df = pd.DataFrame(
    [
        ["Simulation", N_AGENTS, SIM_CO2_T_PER_DAY, SIM_KG_PER_AGENT_DAY],
        ["Reference (inventory, road transport)", REF_POP_IMPLIED, REF_ROAD_T_PER_DAY, REF_KG_PER_CAPITA_DAY],
    ],
    columns=["source", "population", "tCO2_per_day", "kgCO2_per_person_per_day"],
)

display(df)

print(f"Ratio (simulation / reference) = {RATIO_SIM_OVER_REF:.2f}  (e.g., 1.26 = +26%)")
print(
    f"Simulation scaled to reference population ≈ {SCALED_SIM_T_PER_DAY:.1f} tCO2/day "
    f"(reference ≈ {REF_ROAD_T_PER_DAY:.1f} tCO2/day)"
)

,source,population,tCO2_per_day,kgCO2_per_person_per_day
0,Simulation,17107.000000,103.365562,6.042296
1,"Reference (inventory, road transport)",162380.775862,776.484932,4.781877


Ratio (simulation / reference) = 1.26  (e.g., 1.26 = +26%)
Simulation scaled to reference population ≈ 981.2 tCO2/day (reference ≈ 776.5 tCO2/day)
